In [26]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import glob
import cv2
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Basic function
def mean(array):
    return np.mean(np.array(array).flatten())
def mode(array):
    return np.bincount(np.array(array).flatten()).argmax()
def min1(array):
    return np.min(np.array(array).flatten())
def max1(array):
    return np.max(np.array(array).flatten())
def std1(array):
    return np.std(np.array(array).flatten())
def rgb(indir):
    # get values
    RGB_values=[]
    for image_location in glob.glob(indir):
        concentration = image_location.split("/")[-1].split('-')[0]
        image_id=image_location.split("/")[-1]
        # load
        image = cv2.imread(image_location,cv2.IMREAD_COLOR)
        RGB_values.append([image_id,concentration,mean(image[:,:,0]),mean(image[:,:,1]),mean(image[:,:,2]),
                                                  mode(image[:,:,0]),mode(image[:,:,1]),mode(image[:,:,2]),
                                                  min1(image[:,:,0]),min1(image[:,:,1]),min1(image[:,:,2]),
                                                  max1(image[:,:,0]),max1(image[:,:,1]),max1(image[:,:,2]),
                                                  std1(image[:,:,0]),std1(image[:,:,1]),std1(image[:,:,2])
                        ])
    columns_name=["image","concentration",
    "meanB","meanG","meanR",
    "modeB","modeG","modeR",
    "minB","minG","minR",
    "maxB","maxG","maxR",
    "stdB","stdG","stdR"]

    df = pd.DataFrame(RGB_values, columns=columns_name)
    df["batch"]=df["image"].apply(lambda x: x.split(".j")[0].split("_")[1])
    df["concentration"]=df["concentration"].astype(float)
    # get standard deviation 
    # note: this is different with std for each figures, this std for all figures with the same scene (1 concentration=10 figures for ex)
    std_result=[]
    for batch in df["batch"].unique():
        for concentration in df["concentration"].unique():
            df1=df[(df["batch"]==batch) & (df["concentration"]==concentration)]
            std=np.std(df1).to_list()[1:]
            std.insert(0,concentration)
            std.insert(0,batch)
            std_result.append(std)
    std_result=pd.DataFrame(std_result,columns=columns_name)
    return df,std_result

# Section 1: CuSO4

In [27]:
# Feature extraction
indir="data/balance_image/CuSO4"
outdir=indir
# Feature extraction
# raw_image=os.path.join(indir,"raw_image")
# os.system(f"python3 modules/feature_extraction.py -i {raw_image}  -o {outdir}")

In [28]:
%cd ..

/home/nguyen/Desktop


In [ ]:
# EDA
# background
indir=os.path.join(outdir,"result","background","*.jpg")
bg_val,bg_std=rgb(indir)
bg_val.to_csv("data/balance_image/CuSO4/result/EDA/val_background.csv",index=False)
bg_std.to_csv("data/balance_image/CuSO4/result/EDA/std_background.csv",index=False)
# ROI
indir=os.path.join(outdir,"result","raw_roi","*.jpg")
raw_roi_val,raw_roi_std=rgb(indir)
raw_roi_val.to_csv("data/balance_image/CuSO4/result/EDA/val_raw_roi.csv",index=False)
raw_roi_std.to_csv("data/balance_image/CuSO4/result/EDA/std_raw_roi.csv",index=False)
# Delta ROI
indir=os.path.join(outdir,"result","delta_normalized_roi","*.jpg")
delta_roi_val,delta_roi_std=rgb(indir)
delta_roi_val.to_csv("data/balance_image/CuSO4/result/EDA/val_delta_roi.csv",index=False)
delta_roi_std.to_csv("data/balance_image/CuSO4/result/EDA/std_delta_roi.csv",index=False)
# Ratio ROI
indir=os.path.join(outdir,"result","ratio_normalized_roi","*.jpg")
ratio_roi_val,ratio_roi_std=rgb(indir)
ratio_roi_val.to_csv("data/balance_image/CuSO4/result/EDA/val_ratio_roi.csv",index=False)
ratio_roi_std.to_csv("data/balance_image/CuSO4/result/EDA/std_ratio_roi.csv",index=False)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
delta_roi_val

,image,concentration,meanB,meanG,meanR,modeB,modeG,modeR,minB,minG,minR,maxB,maxG,maxR,stdB,stdG,stdR,batch
0,2.0-5_batch3.jpg,2.00,44.562112,0.090224,0.160672,44,0,0,36,0,0,50,2,2,2.364612,0.290275,0.369054,batch3
1,0.25-1_batch3.jpg,0.25,47.210192,23.855792,8.701760,49,26,9,36,12,0,55,31,15,2.648319,2.878103,2.000547,batch3
2,1.75-10_batch3.jpg,1.75,45.701888,0.115056,0.153072,46,0,0,36,0,0,52,4,3,2.557979,0.324361,0.363683,batch3
3,0.75-8_batch3.jpg,0.75,50.561600,8.041200,0.412880,53,7,0,37,0,0,58,15,4,3.189768,2.787615,0.591197,batch3
4,0.5-7_batch1.jpg,0.50,50.437488,12.497200,0.213664,50,14,0,39,1,0,59,19,5,3.177447,2.727884,0.463253,batch1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,0.5-6_batch1.jpg,0.50,53.586640,14.780992,0.321120,55,16,0,41,4,0,61,21,3,3.275479,2.560382,0.511580,batch1
236,1.0-2_batch2.jpg,1.00,51.307296,3.518640,0.390896,50,5,0,41,0,0,66,17,9,2.910318,1.817485,0.617259,batch2
237,0.5-5_batch2.jpg,0.50,50.979728,13.986240,0.223888,52,16,0,40,5,0,60,24,8,2.779517,2.273099,0.469692,batch2
238,0.75-2_batch3.jpg,0.75,51.186928,9.897152,0.287264,54,12,0,37,1,0,58,17,4,3.312384,2.557577,0.506161,batch3


In [ ]:
def evaluation(learning_model,transform_model,x,y_real):
    X_t=transform_model.fit_transform(x)
    y_pred=learning_model.predict(X_t)
    # evaluation
    rmse = np.sqrt(np.mean((y_real - y_pred)**2))
    mae  = np.mean(np.abs(y_real - y_pred))
    pmae = np.mean(np.abs((y_real - y_pred) / y_real)) * 100
    r2   = r2_score(y_real,y_pred)
    return [rmse,mae,pmae,r2]

In [95]:
# Train
# train will be the batch 1
# test will be on batch 2 and 3
def train_test(batch_train,df,degree_poly):    
    batch1=df[df["batch"]=="batch1"]
    batch2=df[df["batch"]=="batch2"]
    batch3=df[df["batch"]=="batch3"]
    # train
    batch1=batch1[batch1["batch"]==batch_train]
    x = batch1.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = batch1["concentration"].values.astype(float)  # target variable
    poly = PolynomialFeatures(degree=degree_poly)
    X_t=poly.fit_transform(x)
    clf = LinearRegression()
    clf.fit(X_t, y)
    # test
    result_all=[]
    # batch 1
    x = batch1.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = batch1["concentration"].values.astype(float)  # target variable
    result=evaluation(clf1D,poly1D,x,y)
    result.insert(0,"batch1")
    result_all.append(result)
    # batch 2
    x = batch2.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = batch2["concentration"].values.astype(float)  # target variable
    result=evaluation(clf,poly,x,y)
    result.insert(0,"batch2")
    result_all.append(result)
    # batch 3
    x = batch3.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = batch3["concentration"].values.astype(float)  # target variable
    result=evaluation(clf,poly,x,y)
    result.insert(0,"batch3")
    result_all.append(result)
    result_all=pd.DataFrame(result_all,columns=["batch","rmse","mae","pmae","r^2"])
    return clf,poly,result_all

In [70]:
# Evaluation metrics
train_test("batch1")

In [74]:
# train on batch1 and predict on batch1,2,3 1D


,batch,rmse,mae,pmae,r^2
0,batch1,0.141604,0.107345,11.397204,0.938890
1,batch2,0.253419,0.213165,30.048314,0.804279
2,batch3,0.248837,0.186900,19.636126,0.811291


In [75]:
# train on batch1 and predict on batch1,2,3 1D
result_2D=[]
# batch 1
x = batch1.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch1["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch1")
result_2D.append(result)
# batch 2
x = batch2.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch2["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch2")
result_2D.append(result)
# batch 3
x = batch3.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch3["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch3")
result_2D.append(result)
pd.DataFrame(result_2D,columns=["batch","rmse","mae","pmae","r^2"])

,batch,rmse,mae,pmae,r^2
0,batch1,6.493809e-13,5.115464e-13,7.653694e-11,1.000000
1,batch2,1.115105e+01,4.950903e+00,1.443951e+03,-377.958958
2,batch3,1.008265e+01,4.020350e+00,1.115823e+03,-308.820402


In [91]:
# retrain batch 2
batch=df[df["batch"]=="batch3"]
x = df.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = df["concentration"].values.astype(float)  # target variable
poly1D = PolynomialFeatures(degree=1)
poly2D = PolynomialFeatures(degree=2)
X_t1D = poly1D.fit_transform(x)
X_t2D = poly2D.fit_transform(x)
clf1D = LinearRegression()
clf2D = LinearRegression()
clf1D.fit(X_t1D, y)
clf2D.fit(X_t2D, y)
# train on batch1 and predict on batch1,2,3 1D
result_1D=[]
# batch 1
x = batch1.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch1["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch1")
result_1D.append(result)
# batch 2
x = batch2.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch2["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch2")
result_1D.append(result)
# batch 3
x = batch3.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch3["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch3")
result_1D.append(result)
result_1D=pd.DataFrame(result_1D,columns=["batch","rmse","mae","pmae","r^2"])
# validate
# train on batch1 and predict on batch1,2,3 1D
result_2D=[]
# batch 1
x = batch1.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch1["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch1")
result_2D.append(result)
# batch 2
x = batch2.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch2["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch2")
result_2D.append(result)
# batch 3
x = batch3.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
y = batch3["concentration"].values.astype(float)  # target variable
result=evaluation(clf2D,poly2D,x,y)
result.insert(0,"batch3")
result_2D.append(result)
result_2D=pd.DataFrame(result_2D,columns=["batch","rmse","mae","pmae","r^2"])

In [92]:
result_1D

,batch,rmse,mae,pmae,r^2
0,batch1,0.318957,0.251185,35.790540,0.689955
1,batch2,0.309319,0.249992,43.839410,0.708408
2,batch3,0.263665,0.200667,28.649977,0.788132


In [93]:
result_2D

,batch,rmse,mae,pmae,r^2
0,batch1,0.318957,0.251185,35.790540,0.689955
1,batch2,0.309319,0.249992,43.839410,0.708408
2,batch3,0.263665,0.200667,28.649977,0.788132
